In [66]:
from sklearn.datasets import make_circles

In [67]:
nSamples = 1000

X, y = make_circles(nSamples, noise = 0.03, random_state=69)

In [68]:
X[:5], y[:10]

(array([[-0.51839217, -0.66522091],
        [ 0.78398275, -0.00098577],
        [ 0.12616247, -0.75963138],
        [-0.70384469, -0.32717022],
        [ 0.67524498, -0.34785267]]),
 array([1, 1, 1, 1, 1, 1, 1, 1, 0, 0]))

In [69]:
import pandas as pd

In [70]:
circles = pd.DataFrame({"X1": X[:, 0], "X2":X[:, 1], "label": y})
circles

,X1,X2,label
0,-0.518392,-0.665221,1
1,0.783983,-0.000986,1
2,0.126162,-0.759631,1
3,-0.703845,-0.327170,1
4,0.675245,-0.347853,1
...,...,...,...
995,0.234010,0.725191,1
996,0.032406,0.785927,1
997,0.722464,-0.268858,1
998,-0.848932,0.541532,0


In [71]:
circles.label.value_counts()

label
1    500
0    500
Name: count, dtype: int64

In [72]:
X.shape, y.shape

X_sample = X[0]
y_sample = y[0]
print(f"Values for one sample of X: {X_sample}")
print(f"Values for one sampele of y: {y_sample}")
print(f"Shapes: {X_sample.shape}   {y_sample.shape}")

Values for one sample of X: [-0.51839217 -0.66522091]
Values for one sampele of y: 1
Shapes: (2,)   ()


In [73]:
import torch

X = torch.from_numpy(X).type(torch.float)
y = torch.from_numpy(y).type(torch.float)

In [74]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=69, test_size = 0.2)

In [75]:
from torch import nn

In [76]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [48]:
class CircleModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.layer1 = nn.Linear(in_features=2, out_features=5) # Because we have two in our data
        self.layer2 = nn.Linear(in_features=5, out_features=1) # Because we gotta predict one y label
    
    def forward(self, x):
        return(self.layer2(self.layer1(x)))

model = CircleModel().to(device)

In [62]:
model = nn.Sequential(nn.Linear(in_features=2, out_features=5), nn.Linear(in_features=5, out_features=1)).to(device)

In [50]:
print(model.state_dict())

OrderedDict({'0.weight': tensor([[ 0.6549,  0.2768],
        [ 0.5878,  0.0740],
        [-0.3756, -0.2457],
        [-0.0677,  0.2856],
        [ 0.1801, -0.5734]]), '0.bias': tensor([ 0.4985, -0.2019,  0.3524, -0.2009,  0.0640]), '1.weight': tensor([[ 0.2836, -0.3962, -0.0394, -0.4308,  0.0764]]), '1.bias': tensor([0.1449])})


In [51]:
untrainedPreds = model(X_test.to(device))

In [78]:
lossFunction = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(params = model.parameters(), lr = 0.1)

In [79]:
def accuracy(yTrue, yPred):
    correct = torch.eq(yTrue, yPred).sum().item()
    acc = (correct / len(yPred)) * 100
    return acc

In [80]:
# Checking to see what forward pass does with raw inputs
yLogits = model(X_test.to(device))[:5]

In [81]:
X_test.shape

torch.Size([200, 2])

In [87]:
torch.manual_seed(69)

In [88]:
epochs = 1000
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_train.to(device), y_train.to(device)

In [89]:
for epoch in range(epochs):
    model.train()
    yLogits = model(X_train).squeeze()
    yPred = torch.round(torch.sigmoid(yLogits)) # Logits to Prediction Probabilities to Labels(0 or 1)

    loss = lossFunction(yLogits, y_train)
    acc = accuracy(yTrue=y_train, yPred=yPred)

    optimizer.zero_grad()
    loss.backward()

    optimizer.step()

    model.eval()
    with torch.inference_mode():
        testLogits = model(X_test).squeeze()
        testPred = torch.round(torch.sigmoid(testLogits))

        testLoss = lossFunction(testLogits, y_test)

        testAcc = accuracy(yTrue=y_test, yPred=testPred)

    if epoch % 10 == 0:
        print(f"Epoch {epoch} || Loss: {loss:.5f} || Accuracy: {acc:.2f} || Test Loss: {testLoss:.5f} || Test Accuracy: {testAcc:.2f}")

Epoch 0 || Loss: 0.49310 || Accuracy: 75.75 || Test Loss: 0.49304 || Test Accuracy: 75.75
Epoch 10 || Loss: 0.49250 || Accuracy: 76.12 || Test Loss: 0.49245 || Test Accuracy: 76.12
Epoch 20 || Loss: 0.49187 || Accuracy: 76.50 || Test Loss: 0.49181 || Test Accuracy: 76.50
Epoch 30 || Loss: 0.49111 || Accuracy: 76.62 || Test Loss: 0.49103 || Test Accuracy: 76.62
Epoch 40 || Loss: 0.49036 || Accuracy: 76.62 || Test Loss: 0.49030 || Test Accuracy: 76.62
Epoch 50 || Loss: 0.48971 || Accuracy: 76.50 || Test Loss: 0.48965 || Test Accuracy: 76.50
Epoch 60 || Loss: 0.48908 || Accuracy: 76.50 || Test Loss: 0.48897 || Test Accuracy: 76.50
Epoch 70 || Loss: 0.48806 || Accuracy: 77.00 || Test Loss: 0.48797 || Test Accuracy: 77.12
Epoch 80 || Loss: 0.48718 || Accuracy: 77.00 || Test Loss: 0.48710 || Test Accuracy: 77.00
Epoch 90 || Loss: 0.48642 || Accuracy: 77.00 || Test Loss: 0.48635 || Test Accuracy: 77.00
Epoch 100 || Loss: 0.48571 || Accuracy: 77.00 || Test Loss: 0.48564 || Test Accuracy: 77.00

## Obviously the model didn't actually fit the original data, so let's bring some non-linearity

In [77]:
class CircleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(in_features=2, out_features=5)
        self.layer2 = nn.Linear(in_features=5, out_features=10)
        self.layer3 = nn.Linear(in_features=10, out_features=1)
        self.relu = nn.ReLU()
    def forward(self, x):
        return self.layer3(self.relu(self.layer2(self.relu(self.layer1(x)))))

model = CircleModel().to(device)
print(model.state_dict())

OrderedDict({'layer1.weight': tensor([[-0.3894, -0.6189],
        [-0.4502,  0.7068],
        [ 0.1336,  0.2179],
        [-0.6595, -0.4644],
        [-0.2354,  0.1106]]), 'layer1.bias': tensor([-0.6222, -0.3047, -0.4233,  0.0020, -0.2631]), 'layer2.weight': tensor([[-0.0310, -0.3030, -0.3070, -0.2609, -0.1531],
        [-0.3530,  0.3750, -0.0888,  0.3848,  0.1393],
        [-0.3787,  0.3095, -0.1230, -0.1714, -0.3712],
        [-0.4446,  0.1280, -0.0977,  0.1741, -0.3670],
        [ 0.3320, -0.3283, -0.0772,  0.0934,  0.2309],
        [ 0.3610,  0.4074, -0.3546,  0.1125, -0.1924],
        [-0.0490, -0.3347,  0.4073, -0.3282,  0.2390],
        [ 0.1572,  0.1453, -0.2418,  0.4065,  0.0983],
        [ 0.0575, -0.3941,  0.1877, -0.0671, -0.2049],
        [ 0.3841,  0.0997, -0.2474, -0.2264, -0.0214]]), 'layer2.bias': tensor([ 0.2497, -0.1143, -0.2552, -0.1531, -0.3341,  0.1595,  0.3462, -0.4210,
         0.1039,  0.2310]), 'layer3.weight': tensor([[ 0.0573, -0.1126,  0.1651,  0.1662,  0.1